## Task 1: Build a Relational Schema

**Step 1:** Create these tables:

* `customers` with columns: `customer_id`, `name`, `email`, `city`, `signup_date`
* `movies` with columns: `movie_id`, `title`, `genre`, `release_year`, `rating` (1-5 scale)
* `rentals` with columns: `rental_id`, `customer_id`, `movie_id`, `rental_date`, `return_date`, `price`

Populate with at least 8 customers, 10 movies, and 25 rentals. Make sure some customers have multiple rentals and some movies are rented multiple times.

In [1]:
import pandas as pd
import numpy as np

# 1. Customers Table (8 entries)
customers = pd.DataFrame({
    "customer_id": [f"C{i:03}" for i in range(1, 9)],
    "name": ["Alice", "Bob", "Charlie", "Diana", "Edward", "Fiona", "George", "Hannah"],
    "email": ["alice@web.com", "bob@mail.com", "charlie@web.com", "diana@mail.com", 
              "edward@web.com", "fiona@mail.com", "george@web.com", "hannah@mail.com"],
    "city": ["Berlin", "Seoul", "Berlin", "Nairobi", "Toronto", "Seoul", "Lima", "Toronto"],
    "signup_date": ["2024-01-10", "2024-02-15", "2024-03-05", "2024-01-20", 
                    "2024-05-12", "2024-04-22", "2024-01-05", "2024-06-01"]
})

# 2. Movies Table (10 entries)
movies = pd.DataFrame({
    "movie_id": [f"M{i:03}" for i in range(1, 11)],
    "title": ["Inception", "The Matrix", "Parasite", "Lion King", "Interstellar", 
              "Pulp Fiction", "Toy Story", "The Godfather", "Spirited Away", "Mad Max"],
    "genre": ["Sci-Fi", "Sci-Fi", "Thriller", "Animation", "Sci-Fi", 
              "Crime", "Animation", "Crime", "Animation", "Action"],
    "release_year": [2010, 1999, 2019, 1994, 2014, 1994, 1995, 1972, 2001, 2015],
    "rating": [5, 5, 4, 3, 5, 4, 2, 5, 5, 4] 
})

# 3. Rentals Table (25 entries)
np.random.seed(42)

# Ensure every customer appears at least once by shuffling the full ID list
initial_customers = customers["customer_id"].tolist()
np.random.shuffle(initial_customers)

# Fill the rest of the 25 slots randomly
remaining_customers = np.random.choice(customers["customer_id"], 17).tolist()
all_rental_customers = initial_customers + remaining_customers

rentals = pd.DataFrame({
    "rental_id": range(1, 26),
    "customer_id": all_rental_customers,
    "movie_id": np.random.choice(movies["movie_id"], 25),
    "rental_date": pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.randint(0, 30, 25), unit='D'),
    "price": np.random.choice([2.99, 3.99, 4.99], 25)
})

rentals["return_date"] = rentals["rental_date"] + pd.to_timedelta(np.random.randint(1, 7, 25), unit='D')

print("Relational schema initialized successfully.")

Relational schema initialized successfully.


**Step 2:** Write and execute these queries using pandas operations:

1. Find all movies rented by a specific customer (requires joining rentals with movies).
2. Find the top 5 most-rented movies (group and count).
3. Compute the total revenue per genre (join rentals with movies, group by genre, sum price).
4. Find customers who have never rented a movie rated below 3 (multi-step filtering).

In [2]:
# Query 1: Find all movies rented by customer 'Alice' (C001)
alice_rentals = rentals[rentals['customer_id'] == 'C001'].merge(movies, on='movie_id')
print("Alice's Rented Movies:\n", alice_rentals[['title', 'genre']])

# Query 2: Find the top 5 most-rented movies
top_5_rented = rentals.groupby('movie_id').size().sort_values(ascending=False).head(5)
top_5_with_titles = top_5_rented.to_frame(name='rental_count').merge(movies, on='movie_id')
print("\nTop 5 Most Rented Movies:\n", top_5_with_titles[['title', 'rental_count']])

# Query 3: Compute total revenue per genre
genre_revenue = rentals.merge(movies, on='movie_id').groupby('genre')['price'].sum()
print("\nTotal Revenue per Genre:\n", genre_revenue)

# Query 4: Find customers who have NEVER rented a movie rated below 3
# Step A: Find customers who DID rent a movie < 3
customers_with_low_ratings = rentals.merge(movies, on='movie_id')
ids_with_low = customers_with_low_ratings[customers_with_low_ratings['rating'] < 3]['customer_id'].unique()
# Step B: Filter main customer list to exclude those IDs
high_standard_customers = customers[~customers['customer_id'].isin(ids_with_low)]
print("\nCustomers who only rent movies rated 3 or higher:\n", high_standard_customers['name'].tolist())

Alice's Rented Movies:
           title   genre
0  Interstellar  Sci-Fi

Top 5 Most Rented Movies:
            title  rental_count
0  Spirited Away             4
1        Mad Max             3
2     The Matrix             3
3   Interstellar             3
4       Parasite             3

Total Revenue per Genre:
 genre
Action       10.97
Animation    34.91
Crime         7.98
Sci-Fi       30.92
Thriller     10.97
Name: price, dtype: float64

Customers who only rent movies rated 3 or higher:
 ['Alice', 'Bob', 'Edward', 'Fiona', 'George', 'Hannah']


**Step 3:** Demonstrate normalization. Show what the data would look like as a single denormalized table (join all three tables). Write a markdown cell explaining what redundancy appears and why the normalized version is preferable for updates.

In [3]:
# Create a denormalized version by joining everything
denormalized_df = rentals.merge(customers, on='customer_id').merge(movies, on='movie_id')

print("Denormalized Table (Sample):")
display(denormalized_df[['rental_id', 'name', 'email', 'title', 'genre']].head())

Denormalized Table (Sample):


,rental_id,name,email,title,genre
0,1,Bob,bob@mail.com,Pulp Fiction,Crime
1,2,Fiona,fiona@mail.com,The Matrix,Sci-Fi
2,3,Alice,alice@web.com,Interstellar,Sci-Fi
3,4,Hannah,hannah@mail.com,Inception,Sci-Fi
4,5,Charlie,charlie@web.com,Mad Max,Action


**Normalization Analysis**
In the **denormalized** table above, we see significant **data redundancy**. For example, if a customer rents five movies, their name, email, and city are repeated five times. Similarly, movie details like genre and release year are repeated every time that movie is rented.

**Why the normalized version is preferable for updates:**

1. **Data Integrity:** If "Alice" changes her email address, in a normalized system, we update it **once** in the `customers` table. In a denormalized system, we would have to find and update every single rental record she ever made. If we miss one, the data becomes inconsistent.

2. **Storage Efficiency:** By storing strings like names and titles only once, we save significant disk space as the dataset grows into the millions.

3. **Insert Anomalies:** In a normalized system, we can add a new movie to the `movies` table even if no one has rented it yet. In a denormalized system, we might not be able to record a movie exists until a rental occurs.

## Task 2: Model the Same Data as Documents

**Step 1:** Create a list of customer documents (Python dictionaries) where each document contains the customer's information and a nested list of their rental history, with movie details embedded in each rental entry.

In [4]:
# Create a list to store our customer "documents"
customer_docs = []

for _, cust in customers.iterrows():
    # Start with the core customer data
    doc = {
        "customer_id": cust["customer_id"],
        "name": cust["name"],
        "email": cust["email"],
        "city": cust["city"],
        "signup_date": cust["signup_date"],
        "rental_history": [] 
    }
    
    # Filter the rentals belonging to this specific customer
    cust_rentals = rentals[rentals["customer_id"] == cust["customer_id"]]
    
    # For each rental, join the movie details and add it to the list
    for _, rent in cust_rentals.iterrows():
        # Get the specific movie details as a dictionary
        movie_info = movies[movies["movie_id"] == rent["movie_id"]].iloc[0].to_dict()
        
        rental_entry = {
            "rental_id": rent["rental_id"],
            "rental_date": str(rent["rental_date"]),
            "price": rent["price"],
            "movie_details": movie_info  # This is the "Nested" part
        }
        doc["rental_history"].append(rental_entry)
        
    customer_docs.append(doc)

# Print the first document formatted as JSON to verify the nesting
import json
print(json.dumps(customer_docs[0], indent=2))

{
  "customer_id": "C001",
  "name": "Alice",
  "email": "alice@web.com",
  "city": "Berlin",
  "signup_date": "2024-01-10",
  "rental_history": [
    {
      "rental_id": 3,
      "rental_date": "2025-01-21 00:00:00",
      "price": 4.99,
      "movie_details": {
        "movie_id": "M005",
        "title": "Interstellar",
        "genre": "Sci-Fi",
        "release_year": 2014,
        "rating": 5
      }
    }
  ]
}


**Step 2:** Execute the same four queries from Task 1 on the document collection:

1. Find all movies rented by a specific customer.
2. Find the top 5 most-rented movies.
3. Compute total revenue per genre.
4. Find customers who have never rented a movie rated below 3.

In [5]:
# 1. Find all movies rented by Alice
alice_doc = next(doc for doc in customer_docs if doc["name"] == "Alice")
alice_movie_list = [r["movie_details"]["title"] for r in alice_doc["rental_history"]]
print("Alice's Movies (Document Model):", alice_movie_list)

# 2. Top 5 most-rented movies
# We must "flatten" the nested data into a single list of titles
all_titles = []
for doc in customer_docs:
    for r in doc["rental_history"]:
        all_titles.append(r["movie_details"]["title"])

top_5_titles = pd.Series(all_titles).value_counts().head(5)
print("\nTop 5 Most Rented (Document Model):\n", top_5_titles)

# 3. Total revenue per genre
genre_revenue_dict = {}
for doc in customer_docs:
    for r in doc["rental_history"]:
        genre = r["movie_details"]["genre"]
        revenue = r["price"]
        genre_revenue_dict[genre] = genre_revenue_dict.get(genre, 0) + revenue

print("\nTotal Revenue per Genre (Document Model):")
for g, rev in genre_revenue_dict.items():
    print(f"{g}: ${rev:.2f}")

# 4. Customers who never rented a movie rated below 3
high_standard_names = []
for doc in customer_docs:
    # Use any() to check if any rental in their history has a low rating
    has_low_rating = any(r["movie_details"]["rating"] < 3 for r in doc["rental_history"])
    if not has_low_rating:
        high_standard_names.append(doc["name"])

print("\nHigh-standard Customers (Document Model):\n", high_standard_names)

Alice's Movies (Document Model): ['Interstellar']

Top 5 Most Rented (Document Model):
 Spirited Away    4
Interstellar     3
Parasite         3
Mad Max          3
Toy Story        3
Name: count, dtype: int64

Total Revenue per Genre (Document Model):
Sci-Fi: $30.92
Crime: $7.98
Thriller: $10.97
Animation: $34.91
Action: $10.97

High-standard Customers (Document Model):
 ['Alice', 'Bob', 'Edward', 'Fiona', 'George', 'Hannah']


**Step 3:** Write a markdown cell comparing the document queries to the relational queries. Which queries were easier to write? Which were harder? For which query pattern does the document model have a clear advantage?

Relational vs. Document Query Comparison

**Which queries were easier to write?**

Finding **all movies for a specific customer** was much easier in the Document Model. Since the rentals are already embedded within the customer object, we simply access the `rental_history` key. There is no need for a `.merge()` or "Join" operation.

**Which were harder?**

The **Top 5 movies** and **Total revenue per genre** were more complex in the Document Model. In the Relational Model, we could perform a single `groupby()` on the rentals table. In the Document Model, we are forced to write nested loops to "extract" the data from every customer document before we can perform any aggregation.

**For which query pattern does the document model have a clear advantage?**

The Document Model is superior for **point queries** or "document-at-a-time" access. If a web application needs to load a user’s "My Account" page with their full rental history, the Document Model can retrieve all that data in a single read. The Relational Model would require the database engine to jump between three different tables and stitch them back together, which is more resource-intensive.

## Task 3: Model Relationships as a Graph

**Step 1:** Create a graph using Python dictionaries where:

* Nodes represent customers, movies, and genres.
* Edges represent relationships: `rented` (customer → movie), `belongs_to` (movie → genre), `lives_in` (customer → city).

**Suggested structure:**
```python
nodes = {
    "c001": {"type": "customer", "name": "Alice"},
    "m001": {"type": "movie", "title": "Inception"},
    "action": {"type": "genre", "name": "Action"},
    # ... more nodes
}

edges = [
    ("c001", "rented", "m001"),
    ("m001", "belongs_to", "action"),
    # ... more edges
]

In [6]:
# 1. Initialize nodes dictionary
nodes = {}

# Add Customers as nodes
for _, row in customers.iterrows():
    nodes[row['customer_id']] = {"type": "customer", "name": row['name'], "city": row['city']}

# Add Movies as nodes
for _, row in movies.iterrows():
    nodes[row['movie_id']] = {"type": "movie", "title": row['title'], "genre": row['genre']}

# Add Genres as nodes (unique list from the movies table)
for g in movies['genre'].unique():
    nodes[g.lower()] = {"type": "genre", "name": g}

# 2. Initialize edges list
edges = []

# Relationship: customer -> rented -> movie
for _, row in rentals.iterrows():
    edges.append((row['customer_id'], "rented", row['movie_id']))

# Relationship: movie -> belongs_to -> genre
for _, row in movies.iterrows():
    edges.append((row['movie_id'], "belongs_to", row['genre'].lower()))

# Relationship: customer -> lives_in -> city (extra metadata hop)
for _, row in customers.iterrows():
    edges.append((row['customer_id'], "lives_in", row['city']))

print(f"Graph initialized with {len(nodes)} nodes and {len(edges)} edges.")

Graph initialized with 23 nodes and 43 edges.


**Step 2:** Write traversal functions to answer:

1. "Which movies did customer X rent?" — one hop from customer through `rented` edges.
2. "Which genres does customer X prefer?" — two hops: customer → movies → genres, then count.
3. "Which customers share a genre preference with customer X?" — three hops: customer X → movies → genres → other customers who rented movies in the same genre.
4. "Recommend movies for customer X" — find movies in their preferred genres that they have not yet rented.

In [7]:
# Q1: Which movies did customer X rent? (1 Hop)
def get_customer_rentals(cust_id):
    return [nodes[target]['title'] for start, rel, target in edges if start == cust_id and rel == "rented"]

# Q2: Which genres does customer X prefer? (2 Hops: Customer -> Movie -> Genre)
def get_customer_genres(cust_id):
    rented_ids = [target for start, rel, target in edges if start == cust_id and rel == "rented"]
    preferred_genres = [target for start, rel, target in edges if start in rented_ids and rel == "belongs_to"]
    # Count occurrences to see preference
    return pd.Series(preferred_genres).value_counts().to_dict()

# Q3: Which customers share a genre preference with customer X? (3 Hops: X -> Genre -> Other Customers)
def get_shared_preference_customers(cust_id):
    my_genres = list(get_customer_genres(cust_id).keys())
    # Find movies in those genres
    movies_in_genres = [start for start, rel, target in edges if rel == "belongs_to" and target in my_genres]
    # Find customers who rented those movies (excluding X)
    peers = [start for start, rel, target in edges if rel == "rented" and target in movies_in_genres and start != cust_id]
    return list(set([nodes[p]['name'] for p in peers]))

# Q4: Recommend movies for customer X (Preferred genres NOT yet rented)
def recommend_movies(cust_id):
    rented_ids = [target for start, rel, target in edges if start == cust_id and rel == "rented"]
    my_genres = list(get_customer_genres(cust_id).keys())
    
    # Find all movies in my preferred genres
    candidate_movies = [start for start, rel, target in edges if rel == "belongs_to" and target in my_genres]
    
    # Filter out movies already rented
    recommendations = [nodes[m]['title'] for m in candidate_movies if m not in rented_ids]
    return list(set(recommendations))

# --- EXECUTION ---
target_id = "C001" # Alice
print(f"Movies rented by Alice: {get_customer_rentals(target_id)}")
print(f"Alice's Genre Preferences: {get_customer_genres(target_id)}")
print(f"Customers with shared interests: {get_shared_preference_customers(target_id)}")
print(f"Recommendations for Alice: {recommend_movies(target_id)}")

Movies rented by Alice: ['Interstellar']
Alice's Genre Preferences: {'sci-fi': 1}
Customers with shared interests: ['Hannah', 'Fiona', 'George']
Recommendations for Alice: ['The Matrix', 'Inception']


**Step 3:** Write a markdown cell explaining why the recommendation query (Task 3, question 4) would be difficult to express in SQL or with documents. What makes the graph model natural for this kind of traversal?

**Graph Traversal vs. SQL/Document Logic**

**Why would the recommendation query be difficult to express in SQL or with documents?**

In **SQL**, a recommendation query requires multiple self-joins or complex subqueries (joining Rentals to Movies to find Genres, then joining back to Rentals to find other Users, then back to Movies to find unrented titles). As the number of "hops" or relationships increases, the SQL code becomes extremely long and the performance drops significantly. In a **Document Model**, this is even harder because you have to scan every single document and perform manual lookups in memory, as the data isn't naturally linked between documents.

**What makes the graph model natural for this kind of traversal?**

The Graph model treats **relationships** as first-class citizens. Instead of searching and matching IDs across tables, we simply "walk" along pre-defined edges. For a recommendation, we follow a path: `Customer -> Rented -> Movie -> Genre -> Movie`. This "pathfinding" is what graphs are built for, making the logic intuitive to write and computationally efficient regardless of how deep the relationships go.

## Task 4: OLTP vs OLAP Workload Analysis

**Step 1:** Create a DataFrame representing a log of 5,000 e-commerce transactions with columns: `transaction_id`, `customer_id`, `product_id`, `amount`, `payment_method`, `timestamp`, `status` (completed/refunded/pending).

In [8]:
import pandas as pd
import numpy as np

# Set seed for reproducibility
np.random.seed(42)

n_transactions = 5000

# Generating synthetic transaction data
data_oltp_olap = {
    "transaction_id": range(10001, 10001 + n_transactions),
    "customer_id": np.random.randint(1, 1001, n_transactions),
    "product_id": np.random.randint(50, 500, n_transactions),
    "amount": np.random.uniform(5.0, 500.0, n_transactions).round(2),
    "payment_method": np.random.choice(["Credit Card", "PayPal", "Bank Transfer", "Crypto"], n_transactions),
    "timestamp": pd.date_range(start="2025-01-01", periods=n_transactions, freq="15min"),
    "status": np.random.choice(["completed", "pending", "refunded"], n_transactions, p=[0.8, 0.15, 0.05])
}

df_transactions = pd.DataFrame(data_oltp_olap)
print(f"Dataset created with {len(df_transactions)} transactions.")

Dataset created with 5000 transactions.


**Step 2:** Simulate OLTP-style operations:

1. Look up a single transaction by its ID (simulate a point query).
2. Insert a new transaction (append a row).
3. Update the status of a transaction from "pending" to "completed."
4. Check that the transaction is valid: the amount must be positive and the status must be one of the allowed values (simulate a consistency check).

In [9]:
# 1. Look up a single transaction by ID
target_id = 10500
single_tx = df_transactions[df_transactions['transaction_id'] == target_id]
print(f"Point Query Result for ID {target_id}:\n", single_tx)

# 2. Insert a new transaction (Append)
new_tx = {
    "transaction_id": 15001, "customer_id": 888, "product_id": 99, 
    "amount": 120.00, "payment_method": "Credit Card", 
    "timestamp": pd.Timestamp.now(), "status": "pending"
}
df_transactions = pd.concat([df_transactions, pd.DataFrame([new_tx])], ignore_index=True)

# 3. Update the status of a transaction
df_transactions.loc[df_transactions['transaction_id'] == 10500, 'status'] = 'completed'

# 4. Consistency Check (Validation)
is_valid = (df_transactions['amount'] > 0).all() and \
           (df_transactions['status'].isin(['completed', 'pending', 'refunded'])).all()

print(f"\nNew transaction count: {len(df_transactions)}")
print(f"Data Consistency Check Passed: {is_valid}")

Point Query Result for ID 10500:
      transaction_id  customer_id  product_id  amount payment_method  \
499           10500          570         203   452.7  Bank Transfer   

              timestamp     status  
499 2025-01-06 04:45:00  completed  

New transaction count: 5001
Data Consistency Check Passed: True


**Step 3:** Simulate OLAP-style operations on the same data:

1. Compute total revenue by payment method.
2. Find the average transaction amount by month.
3. Identify the top 10 customers by total spending.
4. Compute the refund rate (percentage of refunded transactions).

In [10]:
# 1. Total revenue by payment method
revenue_by_method = df_transactions.groupby('payment_method')['amount'].sum()
print("Revenue by Payment Method:\n", revenue_by_method)

# 2. Average transaction amount by month
df_transactions['month'] = df_transactions['timestamp'].dt.to_period('M')
avg_monthly = df_transactions.groupby('month')['amount'].mean()
print("\nAverage Monthly Transaction:\n", avg_monthly)

# 3. Top 10 customers by total spending
top_10_spenders = df_transactions.groupby('customer_id')['amount'].sum().sort_values(ascending=False).head(10)
print("\nTop 10 Spenders:\n", top_10_spenders)

# 4. Compute the refund rate
refund_rate = (df_transactions['status'] == 'refunded').mean() * 100
print(f"\nOverall Refund Rate: {refund_rate:.2f}%")

Revenue by Payment Method:
 payment_method
Bank Transfer    325338.01
Credit Card      311747.31
Crypto           305403.47
PayPal           324914.58
Name: amount, dtype: float64

Average Monthly Transaction:
 month
2025-01    253.390222
2025-02    253.554382
2026-03    120.000000
Freq: M, Name: amount, dtype: float64

Top 10 Spenders:
 customer_id
39     4232.29
664    3477.66
881    3375.05
277    3326.39
958    3316.47
672    3285.45
891    3129.36
840    3091.53
476    3065.32
613    3062.68
Name: amount, dtype: float64

Overall Refund Rate: 4.96%


**Step 4:** Write a markdown cell explaining: Why would OLTP operations typically use row-major storage while OLAP operations use column-major storage? How does the access pattern of each workload type map to the memory layout that serves it best?

**Workload Analysis: OLTP vs. OLAP Storage**

**Why would OLTP operations typically use row-major storage?**

OLTP workloads involve interacting with one record at a time (e.g., searching for a specific transaction or updating a customer's status). In **row-major** storage, all the fields for a single transaction (ID, Amount, Status, etc.) are stored together in memory. This allows the system to read or write the entire record in a single operation, which is highly efficient for transactional integrity.

**How does OLAP benefit from column-major storage?**

OLAP workloads usually involve looking at only one or two columns across millions of rows (e.g., summing the "Amount" column to find total revenue). In **column-major** storage, all values for the "Amount" column are stored contiguously. This allows the CPU to bypass irrelevant data (like "Payment Method" or "Timestamp") and load only the necessary values into the cache, drastically speeding up aggregations.

## Task 5: Choosing the Right Model

For each of the following real-world scenarios, recommend a data model (relational, document, or graph) and a processing paradigm (OLTP or OLAP). Write 3-4 sentences justifying each choice.

1. A hospital patient record system where doctors need to quickly look up a patient's complete medical history.
2. A social network that needs to suggest "friends of friends" connections.
3. A financial reporting system that computes monthly revenue breakdowns across product categories and regions.
4. An IoT platform that receives sensor readings from 10,000 devices and needs to detect anomalies in real time.
5. A content management system where each article has a different set of metadata fields.

#### **1. Hospital Patient Record System**

* **Recommendation:** **Document Model** with an **OLTP** paradigm.
* **Justification:** Patient records are highly "entity-centric," meaning a doctor needs to pull a patient's entire medical history (visits, labs, prescriptions) at once. A document model allows this data to be nested in a single record, enabling fast lookups without expensive joins. OLTP is the correct paradigm because the system must support frequent, real-time updates and individual record retrievals during consultations.

#### **2. Social Network "Friends of Friends" Suggestion**

* **Recommendation:** **Graph Model** with an **OLTP** paradigm.
* **Justification:** This scenario is defined by relationships rather than the data within the entities. A graph model excels at "multi-hop" traversals (User → Friend → Friend), which are computationally expensive in relational or document models. OLTP is used here because the suggestions need to be generated quickly for individual users as they browse the platform.

#### **3. Financial Reporting System (Revenue Breakdowns)**

* **Recommendation:** **Relational Model** with an **OLAP** paradigm.
* **Justification:** Financial data requires high consistency and the ability to aggregate across various dimensions like region and category. The relational model provides a structured schema for these categories, while the OLAP paradigm (ideally with column-major storage) is designed to scan millions of rows to compute sums and averages for monthly reports efficiently.

#### **4. IoT Platform Anomaly Detection (10,000 devices)**

* **Recommendation:** **Relational (Time-Series optimized)** or **Document Model** with an **OLAP** paradigm.
* **Justification:** IoT devices stream high volumes of structured or semi-structured data. Detecting anomalies requires analyzing streams of data points over time to identify patterns. An OLAP approach is necessary to perform complex analytical calculations (like moving averages or standard deviations) across the incoming sensor data in real time or near-real time.

#### **5. Content Management System (Articles with different metadata)**

* **Recommendation:** **Document Model** with an **OLTP** paradigm.
* **Justification:** Articles in a CMS often have unpredictable or evolving schemas (e.g., a "Recipe" article has different fields than a "Tech News" article). A document model is "schema-flexible," allowing each article to store its own unique metadata fields without requiring a database migration. OLTP is preferred as the primary use case is creating, editing, and viewing individual articles.